# Library

In [1]:
import pandas as pd
import numpy as np
import re
from scipy import stats

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

# 경고 무시
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Windows: 'Malgun Gothic', Mac: 'AppleGothic')
plt.rc('font', family='Malgun Gothic') 
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지

In [2]:
data_path = 'D:\VS_Code\서울시_빅데이터활용_경진대회(시각화)/'

# Data Load

In [3]:
df_23 = pd.read_csv(f'{data_path}생필품 농수축산물 가격 정보_23.csv', encoding='cp949')
df_24 = pd.read_csv(f'{data_path}생필품 농수축산물 가격 정보_24.csv', encoding='cp949')
df_2526 = pd.read_csv(f'{data_path}서울시 생필품 농수축산물 가격 정보_2526.csv', encoding='cp949')
df_23.shape, df_24.shape, df_2526.shape

((76559, 14), (98053, 14), (144684, 19))

**23~26 원본 Concat**

In [4]:
df_total = pd.concat([df_23, df_24, df_2526], axis=0, ignore_index=True)
print(df_total.shape)
df_total

(319296, 30)


,일련번호,시장/마트 번호,시장/마트 이름,품목 번호,품목 이름,실판매규격,가격(원),년도-월,비고,시장유형 구분(시장/마트) 코드,...,연월,시장유형코드,시장유형명,자치구코드,자치구,품목코드,품목표준명,품종명,단위,수량명
0,297326,270.0,성대전통시장,70.0,어묵 300g,NaN,0.0,2023-01,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,297349,270.0,성대전통시장,27.0,파프리카 200g,NaN,1180.0,2023-01,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,297348,270.0,성대전통시장,65.0,통조림(참치) 150g,NaN,0.0,2023-01,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,297347,270.0,성대전통시장,24.0,토마토 1kg,NaN,8650.0,2023-01,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,297346,270.0,성대전통시장,21.0,콩나물 500g,NaN,1000.0,2023-01,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319291,530587,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2025-01,1.0,전통시장,320000.0,도봉구,A0053,소금,(CJ 백설)굵은 소금(천일염),1kg,1봉
319292,530586,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2025-01,1.0,전통시장,320000.0,도봉구,A0052,간장,(샘표)진간장 금F3,860ml,1통
319293,530585,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2025-01,1.0,전통시장,320000.0,도봉구,A0051,된장,(청정원)순창 재래식 된장,1kg,1통
319294,530584,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2025-01,1.0,전통시장,320000.0,도봉구,A0050,고추장,(청정원)순창 태양초 찰고추장,1kg,1통


**데이터 복제**

In [5]:
df_23_cp = df_23.copy()
df_24_cp = df_24.copy()
df_2526_cp = df_2526.copy()
df_total_cp = df_total.copy()

In [6]:
df_23_cp.shape, df_24_cp.shape, df_2526_cp.shape, df_total_cp.shape

((76559, 14), (98053, 14), (144684, 19), (319296, 30))

# 1st_Preprocessing

**초기 진행 절차**

1. 24년 시장유형 구분 모두 1로 수정
2. 23,24년 컬럼명과 컬럼순서를 25/26년 기준으로 통일
3. 24년 날짜 형식 '%y-%m'로 수정
4. 시장마트명 ='암사종합시장'의 시장마트유형 코드 '1'로 채우기

**Drop**

23년
- '2023-01'

25/26년
- 품목코드
- 품목표준명
- 품종명
- 단위
- 수량명

## 날짜 & 컬럼

**24년 날짜형식 통일 & 25/26년 기준 컬럼명 통일**

In [7]:
def standardize_df(df):
    df_clean = df.copy()
    
    # 1. 컬럼명 매핑 (기존과 동일)
    column_mapping = {
        '일련번호': '일련번호', '시장/마트 번호': '시장마트번호', '시장/마트 이름': '시장마트명',
        '품목 번호': '품목번호', '품목 이름': '품목명', '실판매규격': '실제판매규격',
        '가격(원)': '가격', '년도-월': '연월', '비고': '비고',
        '시장유형 구분(시장/마트) 코드': '시장유형코드', '시장유형 구분(시장/마트) 이름': '시장유형명',
        '자치구 코드': '자치구코드', '자치구 이름': '자치구', '점검일자': '점검일자'
    }
    df_clean = df_clean.rename(columns=column_mapping)
    
    # 2. 날짜 형식 표준화 로직 통합
    # '년도-월' 또는 '연월' 컬럼이 존재하는 경우 변환
    if '연월' in df_clean.columns:
        # 데이터가 'Jan-24' 형태인지 확인
        if df_clean['연월'].astype(str).str.contains('-').any() and not df_clean['연월'].astype(str).str.contains('202').any():
            df_clean['연월'] = pd.to_datetime(df_clean['연월'], format='%b-%y').dt.strftime('%Y-%m')
    
    # 3. 컬럼 순서 재지정
    desired_order = [
        '일련번호', '시장마트번호', '시장마트명', '품목번호', '품목명', 
        '실제판매규격', '가격', '연월', '비고', '점검일자', 
        '시장유형코드', '시장유형명', '자치구코드', '자치구'
    ]
    return df_clean[[col for col in desired_order if col in df_clean.columns]]

# 4. 이제 깔끔하게 적용
df_23_cp = standardize_df(df_23)
df_24_cp = standardize_df(df_24)

In [8]:
df_23_cp = df_23_cp[df_23_cp['연월']!='2023-01']
df_23_cp['연월'].value_counts().sort_index()

연월
2023-03    6570
2023-04    7870
2023-05    8014
2023-06    8041
2023-07    7690
2023-08    7696
2023-09    7786
2023-10    7613
2023-11    7729
2023-12    7452
Name: count, dtype: int64

**25/26년 컬럼 삭제**

In [9]:
# 2. 불필요한 5개 컬럼 drop
drop_cols = ['품목코드', '품목표준명', '품종명', '단위', '수량명']
df_2526_cp = df_2526_cp.drop(columns=drop_cols)
df_2526_cp

,일련번호,시장마트번호,시장마트명,품목번호,품목명,실제판매규격,가격,연월,비고,점검일자,시장유형코드,시장유형명,자치구코드,자치구
0,1609931,120,망원시장,12,바나나 1송이,NaN,4000,2026-04,NaN,2026-04-13,1,전통시장,440000,마포구
1,1609935,120,망원시장,119,배추(월동) 1포기,NaN,5500,2026-04,NaN,2026-04-13,1,전통시장,440000,마포구
2,1609939,120,망원시장,122,무(월동) 1개,NaN,2000,2026-04,NaN,2026-04-13,1,전통시장,440000,마포구
3,1609940,120,망원시장,15,양파 1망,NaN,3500,2026-04,NaN,2026-04-13,1,전통시장,440000,마포구
4,1609941,120,망원시장,16,상추(적상추) 100g,NaN,600,2026-04,NaN,2026-04-13,1,전통시장,440000,마포구
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144679,530587,251,쌍문시장,50,굵은소금(천일염) 1kg,NaN,6300,2025-01,NaN,2025-01-20,1,전통시장,320000,도봉구
144680,530586,251,쌍문시장,49,간장 1통,NaN,6500,2025-01,NaN,2025-01-20,1,전통시장,320000,도봉구
144681,530585,251,쌍문시장,48,된장 1kg,NaN,6900,2025-01,NaN,2025-01-20,1,전통시장,320000,도봉구
144682,530584,251,쌍문시장,47,고추장 1kg,NaN,14500,2025-01,NaN,2025-01-20,1,전통시장,320000,도봉구


## 결측치 (여기부터 확인)

### 23년

1. 품목명: 194

In [10]:
df_23_cp.info()
print(df_23_cp.isnull().sum())
df_23_cp.head()

<class 'pandas.core.frame.DataFrame'>
Index: 76461 entries, 98 to 76558
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   일련번호    76461 non-null  int64  
 1   시장마트번호  76461 non-null  int64  
 2   시장마트명   76461 non-null  object 
 3   품목번호    76461 non-null  int64  
 4   품목명     76267 non-null  object 
 5   실제판매규격  0 non-null      float64
 6   가격      76461 non-null  int64  
 7   연월      76461 non-null  object 
 8   비고      9995 non-null   object 
 9   점검일자    76461 non-null  object 
 10  시장유형코드  76461 non-null  int64  
 11  시장유형명   76461 non-null  object 
 12  자치구코드   76461 non-null  int64  
 13  자치구     76461 non-null  object 
dtypes: float64(1), int64(6), object(7)
memory usage: 8.8+ MB
일련번호          0
시장마트번호        0
시장마트명         0
품목번호          0
품목명         194
실제판매규격    76461
가격            0
연월            0
비고        66466
점검일자          0
시장유형코드        0
시장유형명         0
자치구코드         0
자치구           0
dtype: int64


,일련번호,시장마트번호,시장마트명,품목번호,품목명,실제판매규격,가격,연월,비고,점검일자,시장유형코드,시장유형명,자치구코드,자치구
98,10922,232,창신골목시장,8,오렌지 1개,NaN,1200,2023-03,NaN,2023-03-27,1,전통시장,110000,종로구
99,10921,232,창신골목시장,7,귤 10개,NaN,16700,2023-03,NaN,2023-03-27,1,전통시장,110000,종로구
100,10920,232,창신골목시장,89,단감 1개(200g),NaN,1200,2023-03,NaN,2023-03-27,1,전통시장,110000,종로구
101,10961,232,창신골목시장,79,"소주 360ml, 1병",NaN,2800,2023-03,NaN,2023-03-27,1,전통시장,110000,종로구
102,10960,232,창신골목시장,76,우유 1L,NaN,2300,2023-03,NaN,2023-03-27,1,전통시장,110000,종로구


In [11]:
df_23_cp[df_23_cp['품목명'].isnull()]

,일련번호,시장마트번호,시장마트명,품목번호,품목명,실제판매규격,가격,연월,비고,점검일자,시장유형코드,시장유형명,자치구코드,자치구
103,10959,232,창신골목시장,75,NaN,NaN,1550,2023-03,NaN,2023-03-27,1,전통시장,110000,종로구
119,10943,232,창신골목시장,47,NaN,NaN,9800,2023-03,NaN,2023-03-27,1,전통시장,110000,종로구
186,15885,233,백학시장,16,NaN,NaN,1000,2023-03,NaN,2023-03-27,1,전통시장,140000,중구
220,17409,239,행당시장상점가,72,NaN,NaN,4800,2023-03,NaN,2023-03-27,1,전통시장,200000,성동구
233,15798,238,성동용답상가시장,69,NaN,NaN,5500,2023-03,NaN,2023-03-27,1,전통시장,200000,성동구
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14009,54044,151,암사종합시장,15,NaN,NaN,1200,2023-04,NaN,2023-04-27,1,전통시장,740000,강동구
14075,54415,205,둔촌역전통시장,59,NaN,NaN,10150,2023-04,NaN,2023-04-27,1,전통시장,740000,강동구
14148,54125,199,서울중앙시장,15,NaN,NaN,3000,2023-04,NaN,2023-04-28,1,전통시장,140000,중구
14228,53922,120,망원시장,15,NaN,NaN,3500,2023-04,NaN,2023-04-28,1,전통시장,440000,마포구


#### 체크1

**방법1 시계열 보간**

최초 등록된 날짜 기준으로, 품목명 결측치를 앞뒤로 채움. ex) 양파로 수집되었다가 x기간을 기준으로 오렌지로 변경된 경우, 앞부분은 '양파' 그리고 뒷부분은 '오렌지'로 채움

In [12]:
# 1. 데이터를 시간순으로 정렬 (매우 중요)
df_23_cp = df_23_cp.sort_values(by=['시장마트명', '품목번호', '점검일자'])

# 2. 시장마트명과 품목번호가 같은 그룹 내에서 품목명의 빈칸을 앞의 값(ffill)으로 채움
# 그 다음, 그래도 남은 빈칸은 뒤의 값(bfill)으로 채움
df_23_cp['품목명'] = df_23_cp.groupby(['시장마트명', '품목번호'])['품목명'].ffill()
df_23_cp['품목명'] = df_23_cp.groupby(['시장마트명', '품목번호'])['품목명'].bfill()

# 3. 결과 확인
remaining_nulls = df_23_cp['품목명'].isnull().sum()
print(f"시계열 보간 후 '품목명' 결측치 잔여량: {remaining_nulls}개")

시계열 보간 후 '품목명' 결측치 잔여량: 0개


In [13]:
# df_23_cp[df_23_cp['품목번호'] == 75].head(20)
df_23_cp[(df_23_cp['품목번호'] == 75) & (df_23_cp['점검일자'] == '2023-03-27')].head(20)

,일련번호,시장마트번호,시장마트명,품목번호,품목명,실제판매규격,가격,연월,비고,점검일자,시장유형코드,시장유형명,자치구코드,자치구
825,1360,279,강남개포시장,75,"소주 360ml, 1병",NaN,1600,2023-03,NaN,2023-03-27,1,전통시장,680000,강남구
723,4397,202,구로시장,75,우유 1L,NaN,1500,2023-03,NaN,2023-03-27,1,전통시장,530000,구로구
746,13801,216,도곡시장,75,우유 1L,NaN,1650,2023-03,NaN,2023-03-27,1,전통시장,680000,강남구
448,4565,257,마포·공덕시장,75,"소주 360ml, 1병",NaN,1550,2023-03,NaN,2023-03-27,1,전통시장,440000,마포구
581,9679,259,목사랑시장(목4동시장),75,"소주 360ml, 1병",NaN,1480,2023-03,NaN,2023-03-27,1,전통시장,470000,양천구
179,15920,233,백학시장,75,우유 1L,NaN,1500,2023-03,NaN,2023-03-27,1,전통시장,140000,중구
317,13997,245,사가정시장,75,우유 1L,NaN,1500,2023-03,NaN,2023-03-27,1,전통시장,260000,중랑구
228,15803,238,성동용답상가시장,75,우유 1L,NaN,1480,2023-03,NaN,2023-03-27,1,전통시장,200000,성동구
415,12453,251,쌍문시장,75,소시지 1kg,NaN,1550,2023-03,NaN,2023-03-27,1,전통시장,320000,도봉구
820,16039,278,영동전통시장,75,우유 1L,NaN,1550,2023-03,NaN,2023-03-27,1,전통시장,680000,강남구


In [14]:
df_23_cp.isnull().sum()

일련번호          0
시장마트번호        0
시장마트명         0
품목번호          0
품목명           0
실제판매규격    76461
가격            0
연월            0
비고        66466
점검일자          0
시장유형코드        0
시장유형명         0
자치구코드         0
자치구           0
dtype: int64

### 24년

1. 시장마트명: 178
2. 품목명: 75
3. 시장유형코드: 1055

In [15]:
df_24_cp.info()
print(df_24_cp.isnull().sum())
df_24_cp.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98053 entries, 0 to 98052
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   일련번호    98053 non-null  int64  
 1   시장마트번호  98053 non-null  int64  
 2   시장마트명   97875 non-null  object 
 3   품목번호    98053 non-null  int64  
 4   품목명     97978 non-null  object 
 5   실제판매규격  0 non-null      float64
 6   가격      98053 non-null  int64  
 7   연월      98053 non-null  object 
 8   비고      9214 non-null   object 
 9   점검일자    98053 non-null  object 
 10  시장유형코드  96998 non-null  float64
 11  시장유형명   98053 non-null  object 
 12  자치구코드   98053 non-null  int64  
 13  자치구     98053 non-null  object 
dtypes: float64(2), int64(5), object(7)
memory usage: 10.5+ MB
일련번호          0
시장마트번호        0
시장마트명       178
품목번호          0
품목명          75
실제판매규격    98053
가격            0
연월            0
비고        88839
점검일자          0
시장유형코드     1055
시장유형명         0
자치구코드         0
자치구           0
dtype: int64

,일련번호,시장마트번호,시장마트명,품목번호,품목명,실제판매규격,가격,연월,비고,점검일자,시장유형코드,시장유형명,자치구코드,자치구
0,297534,207,관악신사시장(신림4동),72,햄 300g,NaN,4500,2024-01,NaN,2024-01-24,1.0,전통시장,620000,관악구
1,297533,207,관악신사시장(신림4동),28,풋고추 100g,NaN,980,2024-01,NaN,2024-01-24,1.0,전통시장,620000,관악구
2,297532,207,관악신사시장(신림4동),5,포도(샤인머스켓) 1kg,NaN,0,2024-01,NaN,2024-01-24,1.0,전통시장,620000,관악구
3,297531,207,관악신사시장(신림4동),27,파프리카 200g,NaN,3500,2024-01,NaN,2024-01-24,1.0,전통시장,620000,관악구
4,297530,207,관악신사시장(신림4동),65,통조림(참치) 150g,NaN,3300,2024-01,NaN,2024-01-24,1.0,전통시장,620000,관악구


**시장유형코드**

1. 시장유형코드 = 2인 '마트'는 25/26년도에만 존재함.
2. 24년은 시장유형코드 = 1인 전통시장만 존재하므로, 결측치를 모두 '1'로 채움.

In [16]:
df_24_cp['시장유형코드'] = df_24_cp['시장유형코드'].fillna(1)
df_24_cp.isnull().sum()

일련번호          0
시장마트번호        0
시장마트명       178
품목번호          0
품목명          75
실제판매규격    98053
가격            0
연월            0
비고        88839
점검일자          0
시장유형코드        0
시장유형명         0
자치구코드         0
자치구           0
dtype: int64

**시장마트명**

1. 시장마트번호 = 276, 72에서 결측치 확인
2. 해당 '시장마트번호'로 '시장마트명' 채우기
3. 시장마트 번호 = 276(남부종합시장), 72(금남시장)

In [17]:
df_24_cp[df_24_cp['시장마트명'].isnull()]['시장마트번호']

86853    276
86854    276
86855    276
86856    276
86857    276
        ... 
89594     72
89595     72
89596     72
89597     72
89598     72
Name: 시장마트번호, Length: 178, dtype: int64

In [18]:
df_24_cp[df_24_cp['시장마트번호'] == 72]

,일련번호,시장마트번호,시장마트명,품목번호,품목명,실제판매규격,가격,연월,비고,점검일자,시장유형코드,시장유형명,자치구코드,자치구
8047,292561,72,금남시장,72,햄 300g,NaN,0,2024-01,NaN,2024-01-31,1.0,전통시장,200000,성동구
8048,292560,72,금남시장,28,풋고추 100g,NaN,2500,2024-01,NaN,2024-01-31,1.0,전통시장,200000,성동구
8049,292559,72,금남시장,5,포도(샤인머스켓) 1kg,NaN,14500,2024-01,NaN,2024-01-31,1.0,전통시장,200000,성동구
8050,292558,72,금남시장,27,파프리카 200g,NaN,1540,2024-01,NaN,2024-01-31,1.0,전통시장,200000,성동구
8051,292557,72,금남시장,65,통조림(참치) 150g,NaN,3550,2024-01,NaN,2024-01-31,1.0,전통시장,200000,성동구
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97966,504164,72,금남시장,5,포도(샤인머스켓) 1kg,NaN,7400,2024-12,NaN,2024-12-31,1.0,전통시장,200000,성동구
97967,504163,72,금남시장,4,복숭아(백도) 1개,NaN,0,2024-12,NaN,2024-12-31,1.0,전통시장,200000,성동구
97968,504162,72,금남시장,3,배(신고) 1개,NaN,3966,2024-12,NaN,2024-12-31,1.0,전통시장,200000,성동구
97969,504161,72,금남시장,2,사과(부사) 1개,NaN,2475,2024-12,NaN,2024-12-31,1.0,전통시장,200000,성동구


**시계열 보간**

In [19]:
# 1. 인사이트를 바탕으로 한 매핑 딕셔너리 생성
market_map = {276: '남부종합시장', 72: '금남시장'}

# 2. 방식 A: loc를 이용한 정밀 타격 (조건이 적을 때 명확함)
# 시장마트명이 결측치이면서 번호가 일치하는 행만 찾아 값을 채웁니다.
for m_id, m_name in market_map.items():
    df_24_cp.loc[
        (df_24_cp['시장마트명'].isnull()) & (df_24_cp['시장마트번호'] == m_id), 
        '시장마트명'
    ] = m_name

# 3. 방식 B: fillna와 map을 이용한 일괄 처리 (확장성이 좋고 속도가 빠름)
# 향후 매핑해야 할 마트가 늘어날 경우 이 방식이 더 효율적입니다.
# df_24_cp['시장마트명'] = df_24_cp['시장마트명'].fillna(df_24_cp['시장마트번호'].map(market_map))

# 4. 결과 검증
print(f"▶ 처리 후 '시장마트명' 결측치 잔여 개수: {df_24_cp['시장마트명'].isnull().sum()}개")

# 데이터가 잘 들어갔는지 확인 (276, 72번 데이터만 추출)
check_results = df_24_cp[df_24_cp['시장마트번호'].isin([276, 72])][['시장마트번호', '시장마트명']].drop_duplicates()
print("▶ 보간 결과 확인:")
print(check_results)

▶ 처리 후 '시장마트명' 결측치 잔여 개수: 0개
▶ 보간 결과 확인:
      시장마트번호   시장마트명
6986     276  남부종합시장
8047      72    금남시장


In [20]:
# 1. 데이터를 시간순으로 정렬 (매우 중요)
df_24_cp = df_24_cp.sort_values(by=['시장마트명', '품목번호', '점검일자'])

# 2. 시장마트명과 품목번호가 같은 그룹 내에서 품목명의 빈칸을 앞의 값(ffill)으로 채움
# 그 다음, 그래도 남은 빈칸은 뒤의 값(bfill)으로 채움
df_24_cp['품목명'] = df_24_cp.groupby(['시장마트명', '품목번호'])['품목명'].ffill()
df_24_cp['품목명'] = df_24_cp.groupby(['시장마트명', '품목번호'])['품목명'].bfill()

# 3. 결과 확인
remaining_nulls = df_24_cp['품목명'].isnull().sum()
print(f"시계열 보간 후 '품목명' 결측치 잔여량: {remaining_nulls}개")

시계열 보간 후 '품목명' 결측치 잔여량: 0개


In [21]:
df_24_cp.isnull().sum()

일련번호          0
시장마트번호        0
시장마트명         0
품목번호          0
품목명           0
실제판매규격    98053
가격            0
연월            0
비고        88839
점검일자          0
시장유형코드        0
시장유형명         0
자치구코드         0
자치구           0
dtype: int64

### 25/26년

1. 시장마트명: 93

In [22]:
df_2526_cp.info()
print(df_2526_cp.isnull().sum())
df_2526_cp.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144684 entries, 0 to 144683
Data columns (total 14 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   일련번호    144684 non-null  int64  
 1   시장마트번호  144684 non-null  int64  
 2   시장마트명   144591 non-null  object 
 3   품목번호    144684 non-null  int64  
 4   품목명     144684 non-null  object 
 5   실제판매규격  0 non-null       float64
 6   가격      144684 non-null  int64  
 7   연월      144684 non-null  object 
 8   비고      14694 non-null   object 
 9   점검일자    144684 non-null  object 
 10  시장유형코드  144684 non-null  int64  
 11  시장유형명   144684 non-null  object 
 12  자치구코드   144684 non-null  int64  
 13  자치구     144684 non-null  object 
dtypes: float64(1), int64(6), object(7)
memory usage: 15.5+ MB
일련번호           0
시장마트번호         0
시장마트명         93
품목번호           0
품목명            0
실제판매규격    144684
가격             0
연월             0
비고        129990
점검일자           0
시장유형코드         0
시장유형명          0
자치구코드         

,일련번호,시장마트번호,시장마트명,품목번호,품목명,실제판매규격,가격,연월,비고,점검일자,시장유형코드,시장유형명,자치구코드,자치구
0,1609931,120,망원시장,12,바나나 1송이,NaN,4000,2026-04,NaN,2026-04-13,1,전통시장,440000,마포구
1,1609935,120,망원시장,119,배추(월동) 1포기,NaN,5500,2026-04,NaN,2026-04-13,1,전통시장,440000,마포구
2,1609939,120,망원시장,122,무(월동) 1개,NaN,2000,2026-04,NaN,2026-04-13,1,전통시장,440000,마포구
3,1609940,120,망원시장,15,양파 1망,NaN,3500,2026-04,NaN,2026-04-13,1,전통시장,440000,마포구
4,1609941,120,망원시장,16,상추(적상추) 100g,NaN,600,2026-04,NaN,2026-04-13,1,전통시장,440000,마포구


1. 모든 결측치의 시장마트번호 = 31
2. 'df_2526_cp'에서 시장마트번호가 31인 것을 찾아 결측치 채우기
3. 시장마트번호 31 = 인왕시장

In [23]:
# 1. 시장마트번호가 31번인 데이터 중 '시장마트명'이 존재하는 진짜 이름 찾기
market_31_names = df_2526_cp[df_2526_cp['시장마트번호'] == 31]['시장마트명'].dropna().unique()

print(f"시장마트번호 31번의 고유 이름: {market_31_names}")

# 2. 이름이 존재한다면 해당 이름으로 93개 결측치 일괄 채우기
if len(market_31_names) > 0:
    real_name = market_31_names[0]
    df_2526_cp.loc[df_2526_cp['시장마트명'].isnull(), '시장마트명'] = real_name
    print(f"\n✅ 성공: 결측치를 '{real_name}'(으)로 모두 복구했습니다!")
else:
    # 3. 만약 전체 데이터(25~26년)에 31번 시장 이름이 아예 없다면 그때 삭제 (사용자님의 제안)
    print("\n⚠️ 31번 시장의 이름을 찾을 수 없어 해당 93개 행을 삭제합니다.")
    df_2526_cp = df_2526_cp.dropna(subset=['시장마트명'])

# 최종 확인
print(f"남은 '시장마트명' 결측치: {df_2526_cp['시장마트명'].isnull().sum()}개")

시장마트번호 31번의 고유 이름: ['인왕시장']

✅ 성공: 결측치를 '인왕시장'(으)로 모두 복구했습니다!
남은 '시장마트명' 결측치: 0개


In [24]:
print(df_23_cp.isnull().sum())
print(df_24_cp.isnull().sum())
df_2526_cp.isnull().sum()

일련번호          0
시장마트번호        0
시장마트명         0
품목번호          0
품목명           0
실제판매규격    76461
가격            0
연월            0
비고        66466
점검일자          0
시장유형코드        0
시장유형명         0
자치구코드         0
자치구           0
dtype: int64
일련번호          0
시장마트번호        0
시장마트명         0
품목번호          0
품목명           0
실제판매규격    98053
가격            0
연월            0
비고        88839
점검일자          0
시장유형코드        0
시장유형명         0
자치구코드         0
자치구           0
dtype: int64


일련번호           0
시장마트번호         0
시장마트명          0
품목번호           0
품목명            0
실제판매규격    144684
가격             0
연월             0
비고        129990
점검일자           0
시장유형코드         0
시장유형명          0
자치구코드          0
자치구            0
dtype: int64

In [25]:
df_23_cp.shape, df_24_cp.shape, df_2526_cp.shape

((76461, 14), (98053, 14), (144684, 14))

## 1차 concat

In [27]:
df_total_cp = pd.concat([df_23_cp, df_24_cp, df_2526_cp], axis=0, ignore_index=True)
print(df_total_cp.shape)
df_total_cp

(319198, 14)


,일련번호,시장마트번호,시장마트명,품목번호,품목명,실제판매규격,가격,연월,비고,점검일자,시장유형코드,시장유형명,자치구코드,자치구
0,48607,264,가리봉시장,1,쌀 20kg 1포,NaN,0,2023-04,NaN,2023-04-25,1.0,전통시장,530000,구로구
1,85115,264,가리봉시장,1,쌀 20kg 1포,NaN,0,2023-05,NaN,2023-05-23,1.0,전통시장,530000,구로구
2,117385,264,가리봉시장,1,쌀 20kg 1포,NaN,0,2023-06,NaN,2023-06-27,1.0,전통시장,530000,구로구
3,155868,264,가리봉시장,1,쌀 20kg 1포,NaN,0,2023-07,"10kg,48000원",2023-07-27,1.0,전통시장,530000,구로구
4,172038,264,가리봉시장,1,쌀 20kg 1포,NaN,0,2023-08,10kg 48000원,2023-08-24,1.0,전통시장,530000,구로구
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319193,530587,251,쌍문시장,50,굵은소금(천일염) 1kg,NaN,6300,2025-01,NaN,2025-01-20,1.0,전통시장,320000,도봉구
319194,530586,251,쌍문시장,49,간장 1통,NaN,6500,2025-01,NaN,2025-01-20,1.0,전통시장,320000,도봉구
319195,530585,251,쌍문시장,48,된장 1kg,NaN,6900,2025-01,NaN,2025-01-20,1.0,전통시장,320000,도봉구
319196,530584,251,쌍문시장,47,고추장 1kg,NaN,14500,2025-01,NaN,2025-01-20,1.0,전통시장,320000,도봉구


In [28]:
df_total_cp.info()
df_total_cp.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319198 entries, 0 to 319197
Data columns (total 14 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   일련번호    319198 non-null  int64  
 1   시장마트번호  319198 non-null  int64  
 2   시장마트명   319198 non-null  object 
 3   품목번호    319198 non-null  int64  
 4   품목명     319198 non-null  object 
 5   실제판매규격  0 non-null       float64
 6   가격      319198 non-null  int64  
 7   연월      319198 non-null  object 
 8   비고      33903 non-null   object 
 9   점검일자    319198 non-null  object 
 10  시장유형코드  319198 non-null  float64
 11  시장유형명   319198 non-null  object 
 12  자치구코드   319198 non-null  int64  
 13  자치구     319198 non-null  object 
dtypes: float64(2), int64(5), object(7)
memory usage: 34.1+ MB


일련번호           0
시장마트번호         0
시장마트명          0
품목번호           0
품목명            0
실제판매규격    319198
가격             0
연월             0
비고        285295
점검일자           0
시장유형코드         0
시장유형명          0
자치구코드          0
자치구            0
dtype: int64

# Feature Engineering

## 반기/분기/4계절

In [29]:
# 에러가 나면 해당 값을 NaT(Not a Time, 결측치)로 처리하고 넘어갑니다.
df_total_cp['점검일자'] = pd.to_datetime(df_total_cp['점검일자'], errors='coerce')

# 2. 반기(Half-year) 생성
df_total_cp['반기'] = df_total_cp['점검일자'].dt.month.apply(lambda x: '상반기' if x <= 6 else '하반기')

# 3. 분기(Quarter) 생성
df_total_cp['분기'] = df_total_cp['점검일자'].dt.quarter

# 4. 계절(Season) 생성
def get_season(month):
    if month in [3, 4, 5]:
        return '봄'
    elif month in [6, 7, 8]:
        return '여름'
    elif month in [9, 10, 11]:
        return '가을'
    else:
        return '겨울'

df_total_cp['계절'] = df_total_cp['점검일자'].dt.month.apply(get_season)

In [33]:
df_total_cp.info()
print(df_total_cp.isnull().sum())
df_total_cp

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319198 entries, 0 to 319197
Data columns (total 17 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   일련번호    319198 non-null  int64         
 1   시장마트번호  319198 non-null  int64         
 2   시장마트명   319198 non-null  object        
 3   품목번호    319198 non-null  int64         
 4   품목명     319198 non-null  object        
 5   실제판매규격  0 non-null       float64       
 6   가격      319198 non-null  int64         
 7   연월      319198 non-null  object        
 8   비고      33903 non-null   object        
 9   점검일자    319198 non-null  datetime64[ns]
 10  시장유형코드  319198 non-null  float64       
 11  시장유형명   319198 non-null  object        
 12  자치구코드   319198 non-null  int64         
 13  자치구     319198 non-null  object        
 14  반기      319198 non-null  object        
 15  분기      319198 non-null  int32         
 16  계절      319198 non-null  object        
dtypes: datetime64[ns](1), float64

,일련번호,시장마트번호,시장마트명,품목번호,품목명,실제판매규격,가격,연월,비고,점검일자,시장유형코드,시장유형명,자치구코드,자치구,반기,분기,계절
0,48607,264,가리봉시장,1,쌀 20kg 1포,NaN,0,2023-04,NaN,2023-04-25,1.0,전통시장,530000,구로구,상반기,2,봄
1,85115,264,가리봉시장,1,쌀 20kg 1포,NaN,0,2023-05,NaN,2023-05-23,1.0,전통시장,530000,구로구,상반기,2,봄
2,117385,264,가리봉시장,1,쌀 20kg 1포,NaN,0,2023-06,NaN,2023-06-27,1.0,전통시장,530000,구로구,상반기,2,여름
3,155868,264,가리봉시장,1,쌀 20kg 1포,NaN,0,2023-07,"10kg,48000원",2023-07-27,1.0,전통시장,530000,구로구,하반기,3,여름
4,172038,264,가리봉시장,1,쌀 20kg 1포,NaN,0,2023-08,10kg 48000원,2023-08-24,1.0,전통시장,530000,구로구,하반기,3,여름
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319193,530587,251,쌍문시장,50,굵은소금(천일염) 1kg,NaN,6300,2025-01,NaN,2025-01-20,1.0,전통시장,320000,도봉구,상반기,1,겨울
319194,530586,251,쌍문시장,49,간장 1통,NaN,6500,2025-01,NaN,2025-01-20,1.0,전통시장,320000,도봉구,상반기,1,겨울
319195,530585,251,쌍문시장,48,된장 1kg,NaN,6900,2025-01,NaN,2025-01-20,1.0,전통시장,320000,도봉구,상반기,1,겨울
319196,530584,251,쌍문시장,47,고추장 1kg,NaN,14500,2025-01,NaN,2025-01-20,1.0,전통시장,320000,도봉구,상반기,1,겨울


## 대표품목명/세부속성/단위

In [34]:
# 품종 및 규격 정보 추출, 보존
def parse_item_details(text):
    if pd.isna(text):
        return '기타', '기본', '규격없음'
    
    text = str(text)
    
    # 1. 품종/상태/산지 추출 (괄호 안의 내용 추출)
    # 예: '쌀(이천쌀)' -> '이천쌀', '고등어(생물)' -> '생물'
    v_match = re.search(r'\((.*?)\)', text)
    variety = v_match.group(1).strip() if v_match else '기본'
    
    # 2. 규격/중량 추출 (숫자와 그 뒤에 붙는 영문/한글 단위)
    # 예: '20kg', '100g', '1.5L', '1마리'
    s_match = re.search(r'(\d+(?:\.\d+)?\s*[a-zA-Z가-힣]+)', text)
    spec = s_match.group(1).strip() if s_match else '규격없음'
    
    # 3. 핵심 품목명 추출 (기존 정규식 활용)
    core_name = re.sub(r'\(.*?\)', '', text)
    core_name = re.sub(r'\d+(?:\.\d+)?\s*[a-zA-Z가-힣]*', '', core_name)
    core_name = re.sub(r'[.,/]', '', core_name)
    core_name = " ".join(core_name.split())
    
    return core_name, variety, spec

# apply 대신 zip을 활용하면 대용량 데이터에서 속도가 훨씬 빠릅니다.
parsed_results = df_total_cp['품목명'].apply(parse_item_details)
df_total_cp['대표품목명'] = [x[0] for x in parsed_results]
df_total_cp['세부속성'] = [x[1] for x in parsed_results]
df_total_cp['단위'] = [x[2] for x in parsed_results]

# 결과 확인
print(df_total_cp[['품목명', '대표품목명', '세부속성', '단위']].head(10))

         품목명 대표품목명 세부속성    단위
0  쌀 20kg 1포     쌀   기본  20kg
1  쌀 20kg 1포     쌀   기본  20kg
2  쌀 20kg 1포     쌀   기본  20kg
3  쌀 20kg 1포     쌀   기본  20kg
4  쌀 20kg 1포     쌀   기본  20kg
5  쌀 20kg 1포     쌀   기본  20kg
6  쌀 20kg 1포     쌀   기본  20kg
7      사과 1개    사과   기본    1개
8      사과 1개    사과   기본    1개
9      사과 1개    사과   기본    1개


In [36]:
df_total_cp.info()
print(df_total_cp.isnull().sum())
df_total_cp.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319198 entries, 0 to 319197
Data columns (total 20 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   일련번호    319198 non-null  int64         
 1   시장마트번호  319198 non-null  int64         
 2   시장마트명   319198 non-null  object        
 3   품목번호    319198 non-null  int64         
 4   품목명     319198 non-null  object        
 5   실제판매규격  0 non-null       float64       
 6   가격      319198 non-null  int64         
 7   연월      319198 non-null  object        
 8   비고      33903 non-null   object        
 9   점검일자    319198 non-null  datetime64[ns]
 10  시장유형코드  319198 non-null  float64       
 11  시장유형명   319198 non-null  object        
 12  자치구코드   319198 non-null  int64         
 13  자치구     319198 non-null  object        
 14  반기      319198 non-null  object        
 15  분기      319198 non-null  int32         
 16  계절      319198 non-null  object        
 17  대표품목명   319198 non-null  obje

(319198, 20)

## 상세품목명/보정가격

In [37]:
# 세부 품종별 가격 보정 및 통일화
def calculate_standard_price(row):
    item = row['대표품목명']
    variety = row['세부속성']
    spec = row['단위']
    price = row['가격']
    
    # 1. 세부 품목명 생성 (예: '쌀_이천쌀', '고등어_생물')
    detailed_nm = f"{item}_{variety}" if variety != '기본' else item
    
    # 2. 규격에서 숫자만 추출 (예: '20kg' -> 20.0, '1마리' -> 1.0)
    num_match = re.search(r'(\d+(?:\.\d+)?)', spec)
    amount = float(num_match.group(1)) if num_match else 1.0
    
    adj_price = price
    is_valid = True
    
    # 3. 품목별 자동 환산 로직 (단위가 다르면 1단위 가격으로 나눈 뒤, 목표 단위 곱하기)
    if item == '쌀':
        # 목표: 모두 '10kg' 가격으로 보정
        if 'kg' in spec:
            adj_price = (price / amount) * 10
        else:
            is_valid = False
            
    elif item in ['소고기', '돼지고기', '닭고기']:
        # 목표: 모두 '100g' 가격으로 보정
        if 'kg' in spec:
            adj_price = (price / (amount * 1000)) * 100 
        elif 'g' in spec:
            adj_price = (price / amount) * 100
        else:
            is_valid = False
            
    elif item in ['고등어', '갈치', '조기']:
        # 목표: 모두 '1마리' 가격으로 보정
        if '손' in spec: # 1손 = 2마리
            adj_price = (price / (amount * 2)) * 1
        elif '마리' in spec:
            adj_price = (price / amount) * 1
        else:
            is_valid = False
            
    elif item in ['배추', '무', '상추', '고구마']:
        # 채소류는 품종(여름, 가을, 월동 등) 분리가 핵심이므로 규격은 그대로 수용 (필요시 환산 추가)
        adj_price = price
        
    return detailed_nm, adj_price, is_valid

# 보정 함수 적용
pricing_results = df_total_cp.apply(calculate_standard_price, axis=1)

df_total_cp['상세품목명'] = [r[0] for r in pricing_results]
df_total_cp['보정가격'] = [r[1] for r in pricing_results]
df_total_cp['is_valid'] = [r[2] for r in pricing_results]

# 최종 정제된 데이터셋
df_total_cp = df_total_cp[df_total_cp['is_valid'] == True]

# 결과 확인 (이천쌀과 오대쌀이 별도로 10kg 단위 가격으로 보정된 것 확인)
sample_check = df_total_cp[df_total_cp['대표품목명'] == '쌀'][['품목명', '상세품목명', '가격', '보정가격']].sample(10)
print(sample_check)

                    품목명  상세품목명     가격     보정가격
305059   쌀(이천쌀) 20kg 1포  쌀_이천쌀  99000  49500.0
23549         쌀 20kg 1포      쌀  69800  34900.0
218589    쌀(오대쌀) 4kg 1포  쌀_오대쌀  27000  67500.0
235064    쌀(오대쌀) 4kg 1포  쌀_오대쌀  29000  72500.0
275628   쌀(오대쌀) 20kg 1포  쌀_오대쌀  95800  47900.0
295628    쌀(이천쌀) 4kg 1포  쌀_이천쌀  25900  64750.0
53099         쌀 20kg 1포      쌀  59800  29900.0
231366   쌀(이천쌀) 10kg 1포  쌀_이천쌀  47800  47800.0
239170    쌀(오대쌀) 4kg 1포  쌀_오대쌀  24800  62000.0
220181   쌀(이천쌀) 10kg 1포  쌀_이천쌀  55900  55900.0


In [39]:
df_total_cp.info()
print(df_total_cp.isnull().sum())
df_total_cp.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319198 entries, 0 to 319197
Data columns (total 23 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   일련번호      319198 non-null  int64         
 1   시장마트번호    319198 non-null  int64         
 2   시장마트명     319198 non-null  object        
 3   품목번호      319198 non-null  int64         
 4   품목명       319198 non-null  object        
 5   실제판매규격    0 non-null       float64       
 6   가격        319198 non-null  int64         
 7   연월        319198 non-null  object        
 8   비고        33903 non-null   object        
 9   점검일자      319198 non-null  datetime64[ns]
 10  시장유형코드    319198 non-null  float64       
 11  시장유형명     319198 non-null  object        
 12  자치구코드     319198 non-null  int64         
 13  자치구       319198 non-null  object        
 14  반기        319198 non-null  object        
 15  분기        319198 non-null  int32         
 16  계절        319198 non-null  object     

(319198, 23)

## 보정가격 0 이하 & 결측치 제거

In [40]:
# 1. 원본 데이터의 크기 확인 (비교 기준)
print(f"필터링 전 데이터 개수: {len(df_total_cp)}")

df_filtered = df_total_cp[df_total_cp['보정가격'] > 0]

# Null 데이터 제거
df_filtered.dropna(subset='보정가격', inplace=True)

# 3. 전후 데이터 비교
print(f"필터링 후 데이터 개수: {len(df_filtered)}")
print(f"제외된 데이터 개수: {len(df_total_cp) - len(df_filtered)}")

필터링 전 데이터 개수: 319198
필터링 후 데이터 개수: 275366
제외된 데이터 개수: 43832


In [41]:
df_total_cp = df_filtered

df_total_cp.info()
print(df_total_cp.isnull().sum())
df_total_cp.shape

<class 'pandas.core.frame.DataFrame'>
Index: 275366 entries, 7 to 319197
Data columns (total 23 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   일련번호      275366 non-null  int64         
 1   시장마트번호    275366 non-null  int64         
 2   시장마트명     275366 non-null  object        
 3   품목번호      275366 non-null  int64         
 4   품목명       275366 non-null  object        
 5   실제판매규격    0 non-null       float64       
 6   가격        275366 non-null  int64         
 7   연월        275366 non-null  object        
 8   비고        32549 non-null   object        
 9   점검일자      275366 non-null  datetime64[ns]
 10  시장유형코드    275366 non-null  float64       
 11  시장유형명     275366 non-null  object        
 12  자치구코드     275366 non-null  int64         
 13  자치구       275366 non-null  object        
 14  반기        275366 non-null  object        
 15  분기        275366 non-null  int32         
 16  계절        275366 non-null  object        
 

(275366, 23)

In [42]:
df_total_cp[df_total_cp['시장유형코드'] == 2]

,일련번호,시장마트번호,시장마트명,품목번호,품목명,실제판매규격,가격,연월,비고,점검일자,...,자치구,반기,분기,계절,대표품목명,세부속성,단위,상세품목명,보정가격,is_valid
174972,1612550,310,이마트(용산점),163,일회용컵,NaN,860,2026-04,이상없음,2026-04-13,...,용산구,상반기,2,봄,일회용컵,기본,규격없음,일회용컵,860.0,True
174973,1612549,310,이마트(용산점),162,일회용접시,NaN,1180,2026-04,이상없음,2026-04-13,...,용산구,상반기,2,봄,일회용접시,기본,규격없음,일회용접시,1180.0,True
174974,1612548,310,이마트(용산점),161,섬유유연제,NaN,19600,2026-04,이상없음,2026-04-13,...,용산구,상반기,2,봄,섬유유연제,기본,규격없음,섬유유연제,19600.0,True
174975,1612547,310,이마트(용산점),160,고무장갑,NaN,6480,2026-04,이상없음,2026-04-13,...,용산구,상반기,2,봄,고무장갑,기본,규격없음,고무장갑,6480.0,True
174976,1612546,310,이마트(용산점),159,주방세제,NaN,10900,2026-04,이상없음,2026-04-13,...,용산구,상반기,2,봄,주방세제,기본,규격없음,주방세제,10900.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313811,549793,295,홈플러스(강서점),144,포도(샤인머스켓) 2kg,NaN,23316,2025-01,1.2kg 13990,2025-01-31,...,강서구,상반기,1,겨울,포도,샤인머스켓,2kg,포도_샤인머스켓,23316.0,True
313812,549790,295,홈플러스(강서점),3,배(신고) 1개,NaN,4975,2025-01,NaN,2025-01-31,...,강서구,상반기,1,겨울,배,신고,1개,배_신고,4975.0,True
313813,549787,295,홈플러스(강서점),2,사과(부사) 1개,NaN,3498,2025-01,NaN,2025-01-31,...,강서구,상반기,1,겨울,사과,부사,1개,사과_부사,3498.0,True
313814,549785,295,홈플러스(강서점),111,쌀(오대쌀) 10kg 1포,NaN,42900,2025-01,NaN,2025-01-31,...,강서구,상반기,1,겨울,쌀,오대쌀,10kg,쌀_오대쌀,42900.0,True


In [43]:
df_total_cp[(df_total_cp['세부속성'] == '기본') & (df_total_cp['단위'] == '규격없음')]
# 일회용컵	

,일련번호,시장마트번호,시장마트명,품목번호,품목명,실제판매규격,가격,연월,비고,점검일자,...,자치구,반기,분기,계절,대표품목명,세부속성,단위,상세품목명,보정가격,is_valid
174972,1612550,310,이마트(용산점),163,일회용컵,NaN,860,2026-04,이상없음,2026-04-13,...,용산구,상반기,2,봄,일회용컵,기본,규격없음,일회용컵,860.0,True
174973,1612549,310,이마트(용산점),162,일회용접시,NaN,1180,2026-04,이상없음,2026-04-13,...,용산구,상반기,2,봄,일회용접시,기본,규격없음,일회용접시,1180.0,True
174974,1612548,310,이마트(용산점),161,섬유유연제,NaN,19600,2026-04,이상없음,2026-04-13,...,용산구,상반기,2,봄,섬유유연제,기본,규격없음,섬유유연제,19600.0,True
174975,1612547,310,이마트(용산점),160,고무장갑,NaN,6480,2026-04,이상없음,2026-04-13,...,용산구,상반기,2,봄,고무장갑,기본,규격없음,고무장갑,6480.0,True
174976,1612546,310,이마트(용산점),159,주방세제,NaN,10900,2026-04,이상없음,2026-04-13,...,용산구,상반기,2,봄,주방세제,기본,규격없음,주방세제,10900.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197339,1559324,300,이마트(왕십리점),150,초콜릿,NaN,6380,2026-03,롯데ABC 187g/ 이상없슴,2026-03-25,...,성동구,상반기,1,봄,초콜릿,기본,규격없음,초콜릿,6380.0,True
197340,1559325,300,이마트(왕십리점),151,캔디,NaN,3580,2026-03,스카치캔디 317g / 이상없슴,2026-03-25,...,성동구,상반기,1,봄,캔디,기본,규격없음,캔디,3580.0,True
197341,1559326,300,이마트(왕십리점),152,전지분유,NaN,19900,2026-03,서울우유 1kg/ 이상없슴,2026-03-25,...,성동구,상반기,1,봄,전지분유,기본,규격없음,전지분유,19900.0,True
197346,1559338,300,이마트(왕십리점),153,휴지,NaN,19900,2026-03,크리넥스 30/30/이상없슴,2026-03-25,...,성동구,상반기,1,봄,휴지,기본,규격없음,휴지,19900.0,True


## Z-score 결측치 채우기

In [44]:
# 1. '세부 품목명(detailed_nm)' 기준으로 그룹화하여 '보정 가격(adj_price)'의 Z-score 계산
df_total_cp['z_score'] = df_total_cp.groupby('상세품목명')['보정가격'].transform(
    lambda x: stats.zscore(x, nan_policy='omit')
)

# 2. 결측치 처리 (데이터가 1개이거나, 모든 가격이 동일하여 분산이 0인 경우)
df_total_cp['z_score'] = df_total_cp['z_score'].fillna(0)

# 1. 이상치 인덱스 추출 (Z-score 절대값이 3 초과)
threshold = 3
outlier_idx = df_total_cp[np.abs(df_total_cp['z_score']) > threshold].index

# 2. 삭제(Drop) 대신 결측치(NaN)로 덮어쓰기
df_total_cp.loc[outlier_idx, '보정가격'] = np.nan
df_total_cp.loc[outlier_idx, '가격'] = np.nan 

# 3. 처리 결과 확인
print(f"✅ 총 {len(outlier_idx):,}개의 입력 오류 의심 데이터를 결측치(NaN)로 변경 완료!")

✅ 총 2,391개의 입력 오류 의심 데이터를 결측치(NaN)로 변경 완료!


In [45]:
df_total_cp.info()
print(df_total_cp.isnull().sum())
df_total_cp.shape

<class 'pandas.core.frame.DataFrame'>
Index: 275366 entries, 7 to 319197
Data columns (total 24 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   일련번호      275366 non-null  int64         
 1   시장마트번호    275366 non-null  int64         
 2   시장마트명     275366 non-null  object        
 3   품목번호      275366 non-null  int64         
 4   품목명       275366 non-null  object        
 5   실제판매규격    0 non-null       float64       
 6   가격        272975 non-null  float64       
 7   연월        275366 non-null  object        
 8   비고        32549 non-null   object        
 9   점검일자      275366 non-null  datetime64[ns]
 10  시장유형코드    275366 non-null  float64       
 11  시장유형명     275366 non-null  object        
 12  자치구코드     275366 non-null  int64         
 13  자치구       275366 non-null  object        
 14  반기        275366 non-null  object        
 15  분기        275366 non-null  int32         
 16  계절        275366 non-null  object        
 

(275366, 24)

In [46]:
drop_cols = ['is_valid', '시장마트번호', '실제판매규격', ]

df_total_cp.drop(columns = drop_cols, inplace= True)
df_total_cp.shape

(275366, 21)

## 대/중/소

In [47]:
# 1. 딕셔너리 생성 (기존과 동일하게 유지 - 유지보수성 확보)
category_dict = {
    # --- 농축수산물 ---
    '쌀': ('농축수산물', '농산물', '곡류'), '콩': ('농축수산물', '농산물', '곡류'), '팥': ('농축수산물', '농산물', '곡류'),
    '상추': ('농축수산물', '농산물', '채소'), '무': ('농축수산물', '농산물', '채소'), '애호박': ('농축수산물', '농산물', '채소'),
    '오이': ('농축수산물', '농산물', '채소'), '양파': ('농축수산물', '농산물', '채소'), '깻잎': ('농축수산물', '농산물', '채소'),
    '당근': ('농축수산물', '농산물', '채소'), '파프리카': ('농축수산물', '농산물', '채소'), '시금치': ('농축수산물', '농산물', '채소'),
    '토마토': ('농축수산물', '농산물', '채소'), '깐마늘': ('농축수산물', '농산물', '채소'), '대파': ('농축수산물', '농산물', '채소'),
    '감자': ('농축수산물', '농산물', '채소'), '고구마': ('농축수산물', '농산물', '채소'), '콩나물': ('농축수산물', '농산물', '채소'),
    '버섯': ('농축수산물', '농산물', '채소'), '풋고추': ('농축수산물', '농산물', '채소'), '청양고추': ('농축수산물', '농산물', '채소'),
    '양배추': ('농축수산물', '농산물', '채소'), '꽈리고추': ('농축수산물', '농산물', '채소'), '방울토마토': ('농축수산물', '농산물', '채소'),
    '배추': ('농축수산물', '농산물', '채소'), '브로콜리': ('농축수산물', '농산물', '채소'), '부추': ('농축수산물', '농산물', '채소'),
    '쪽파': ('농축수산물', '농산물', '채소'), '가지': ('농축수산물', '농산물', '채소'), '생강': ('농축수산물', '농산물', '채소'),
    '미나리': ('농축수산물', '농산물', '채소'), '도라지': ('농축수산물', '농산물', '채소'), '갓': ('농축수산물', '농산물', '채소'),
    '열무': ('농축수산물', '농산물', '채소'), '호박': ('농축수산물', '농산물', '채소'), '마늘': ('농축수산물', '농산물', '채소'),
    '파': ('농축수산물', '농산물', '채소'), '붉은고추': ('농축수산물', '농산물', '채소'),
    '배': ('농축수산물', '농산물', '과일'), '바나나': ('농축수산물', '농산물', '과일'), '사과': ('농축수산물', '농산물', '과일'),
    '참외': ('농축수산물', '농산물', '과일'), '수박': ('농축수산물', '농산물', '과일'), '오렌지': ('농축수산물', '농산물', '과일'),
    '귤': ('농축수산물', '농산물', '과일'), '단감': ('농축수산물', '농산물', '과일'), '포도': ('농축수산물', '농산물', '과일'),
    '딸기': ('농축수산물', '농산물', '과일'), '골드키위': ('농축수산물', '농산물', '과일'), '복숭아': ('농축수산물', '농산물', '과일'),
    '대추': ('농축수산물', '농산물', '과일'), '밤': ('농축수산물', '농산물', '과일'), '감': ('농축수산물', '농산물', '과일'),
    '소고기': ('농축수산물', '축산물', '정육'), '돼지고기': ('농축수산물', '축산물', '정육'), '닭고기': ('농축수산물', '축산물', '정육'),
    '계란': ('농축수산물', '축산물', '알류'),
    '고등어': ('농축수산물', '수산물', '생선류'), '명태': ('농축수산물', '수산물', '생선류'), '갈치': ('농축수산물', '수산물', '생선류'), 
    '조기': ('농축수산물', '수산물', '생선류'), '조개': ('농축수산물', '수산물', '해산물'), '오징어': ('농축수산물', '수산물', '해산물'), 
    '새우': ('농축수산물', '수산물', '해산물'), '굴': ('농축수산물', '수산물', '해산물'), '낙지': ('농축수산물', '수산물', '해산물'), 
    '전복': ('농축수산물', '수산물', '해산물'), '꽃게': ('농축수산물', '수산물', '해산물'),
    '마른멸치': ('농축수산물', '수산물', '건어물/해조류'), '맛김': ('농축수산물', '수산물', '건어물/해조류'),

    # --- 가공식품 ---
    '설탕': ('가공식품', '조미료', '소스/오일'), '마요네즈': ('가공식품', '조미료', '소스/오일'), '식초': ('가공식품', '조미료', '소스/오일'),
    '식용유': ('가공식품', '조미료', '소스/오일'), '케찹': ('가공식품', '조미료', '소스/오일'), '간장': ('가공식품', '조미료', '소스/오일'),
    '참기름': ('가공식품', '조미료', '소스/오일'), '새우젓': ('가공식품', '조미료', '젓갈류'), '멸치액젓': ('가공식품', '조미료', '젓갈류'),
    '부침가루': ('가공식품', '조미료', '가루/장류'), '고춧가루': ('가공식품', '조미료', '가루/장류'), '된장': ('가공식품', '조미료', '가루/장류'),
    '고추장': ('가공식품', '조미료', '가루/장류'), '밀가루': ('가공식품', '조미료', '가루/장류'), '굵은소금': ('가공식품', '조미료', '가루/장류'),
    '소금': ('가공식품', '조미료', '가루/장류'), '라면': ('가공식품', '면/빵류', '면류'), '컵라면': ('가공식품', '면/빵류', '면류'), 
    '국수': ('가공식품', '면/빵류', '면류'), '빵': ('가공식품', '면/빵류', '빵류'),
    '통조림': ('가공식품', '간편식', '간편조리'), '즉석밥': ('가공식품', '간편식', '간편조리'), '어묵': ('가공식품', '간편식', '간편조리'),
    '만두': ('가공식품', '간편식', '간편조리'), '김치': ('가공식품', '간편식', '간편조리'), '햄': ('가공식품', '간편식', '가공육'),
    '소시지': ('가공식품', '간편식', '가공육'), '두부': ('가공식품', '간편식', '두부류'),
    '우유': ('가공식품', '유제품/간식', '유제품'), '치즈': ('가공식품', '유제품/간식', '유제품'), '분유': ('가공식품', '유제품/간식', '유제품'),
    '에너지바': ('가공식품', '유제품/간식', '간식류'), '초콜릿': ('가공식품', '유제품/간식', '간식류'), '캔디': ('가공식품', '유제품/간식', '간식류'),

    # --- 음료/주류 ---
    '사이다': ('음료/주류', '음료', '탄산/생수'), '콜라': ('음료/주류', '음료', '탄산/생수'), '생수': ('음료/주류', '음료', '탄산/생수'),
    '소주': ('음료/주류', '주류', '주류'), '맥주': ('음료/주류', '주류', '주류'),

    # --- 생필품 ---
    '비누': ('생필품', '위생용품', '바디/헤어'), '샴푸': ('생필품', '위생용품', '바디/헤어'), '바디워시': ('생필품', '위생용품', '바디/헤어'),
    '칫솔': ('생필품', '위생용품', '구강용품'), '치약': ('생필품', '위생용품', '구강용품'),
    '세제': ('생필품', '생활잡화', '세제/세정'), '주방세제': ('생필품', '생활잡화', '세제/세정'), '섬유유연제': ('생필품', '생활잡화', '세제/세정'),
    '위생백': ('생필품', '생활잡화', '주방잡화'), '고무장갑': ('생필품', '생활잡화', '주방잡화'), '일회용컵': ('생필품', '생활잡화', '주방잡화'),
    '휴지': ('생필품', '위생용품', '지류/물티슈'), '물티슈': ('생필품', '위생용품', '지류/물티슈'), 
    '기저귀': ('생필품', '위생용품', '위생용품'), '여성용품': ('생필품', '위생용품', '위생용품')
}

# 2. 딕셔너리를 데이터프레임으로 변환 (조인 키를 'base_nm'으로 설정)
df_category = pd.DataFrame.from_dict(
    category_dict, orient='index', columns=['대분류', '중분류', '소분류']
).reset_index().rename(columns={'index': 'base_nm'})

# 3. 데이터프레임에서 '_' 기준 앞부분만 잘라내어 임시 컬럼(base_nm) 생성
df_total_cp['base_nm'] = df_total_cp['상세품목명'].apply(lambda x: str(x).split('_')[0])

# 4. 생성된 기본 품목명(base_nm)을 기준으로 병합
df_total_cp = pd.merge(df_total_cp, df_category, on='base_nm', how='left')

# 5. 사전에 없는 품목은 '기타'로 채우기
df_total_cp[['대분류', '중분류', '소분류']] = df_total_cp[['대분류', '중분류', '소분류']].fillna('기타')

# 6. 목적을 다한 임시 컬럼(base_nm) 삭제로 데이터프레임 최적화
df_total_cp = df_total_cp.drop(columns=['base_nm'])

# 결과 확인
print(df_total_cp[['상세품목명', '대분류', '중분류', '소분류']].sample(10))

           상세품목명    대분류  중분류    소분류
57484         당근  농축수산물  농산물     채소
104806       식용유   가공식품  조미료  소스/오일
106403       식용유   가공식품  조미료  소스/오일
271909      청양고추  농축수산물  농산물     채소
13328         대파  농축수산물  농산물     채소
34806     소고기_수입  농축수산물  축산물     정육
35306        깐마늘  농축수산물  농산물     채소
172972        생수  음료/주류   음료  탄산/생수
147090  고구마_밤고구마  농축수산물  농산물     채소
271236        식초   가공식품  조미료  소스/오일


In [49]:
df_total_cp.info()
print(df_total_cp.isnull().sum())
df_total_cp.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 275366 entries, 0 to 275365
Data columns (total 24 columns):
 #   Column   Non-Null Count   Dtype         
---  ------   --------------   -----         
 0   일련번호     275366 non-null  int64         
 1   시장마트명    275366 non-null  object        
 2   품목번호     275366 non-null  int64         
 3   품목명      275366 non-null  object        
 4   가격       272975 non-null  float64       
 5   연월       275366 non-null  object        
 6   비고       32549 non-null   object        
 7   점검일자     275366 non-null  datetime64[ns]
 8   시장유형코드   275366 non-null  float64       
 9   시장유형명    275366 non-null  object        
 10  자치구코드    275366 non-null  int64         
 11  자치구      275366 non-null  object        
 12  반기       275366 non-null  object        
 13  분기       275366 non-null  int32         
 14  계절       275366 non-null  object        
 15  대표품목명    275366 non-null  object        
 16  세부속성     275366 non-null  object        
 17  단위       2

(275366, 24)

# 2nd_Preprocessing

**Drop**
1. 자치구코드
2. 시장유형코드
3. 시장유형명

**결측치**

가격, 보정가격 -> 개수 = 2391로 동일